# TF-IDF Baseline — Spam Classification

No GPU needed for this one - TF-IDF + Logistic Regression trains in seconds.
Only Drive is needed, for the dataset. Run each cell with Shift+Enter.

## 1. Mount Drive (for the dataset)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install dependencies

In [2]:
!pip install -q scikit-learn joblib

## 3. Run the TF-IDF baseline

In [3]:
import os
import json
import time
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/llm_finetune_project/data/spam"
RESULTS_DIR = "/content/drive/MyDrive/llm_finetune_project/results/spam"
CHECKPOINT_DIR = "/content/drive/MyDrive/llm_finetune_project/checkpoints/spam"

MAX_FEATURES = 5000  # vocabulary size cap for the TF-IDF vectorizer


# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# TF-IDF doesn't use a separate validation set the way a neural network does
# (no epochs to tune against) - so train+val are combined for fitting, and
# test stays held out exactly like the other three fine-tuning scripts.
fit_texts = pd.concat([train_df["text"], val_df["text"]]).astype(str).tolist()
fit_labels = pd.concat([train_df["label"], val_df["label"]]).tolist()

test_texts = test_df["text"].astype(str).tolist()
test_labels = test_df["label"].tolist()


# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
start_time = time.time()

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, stop_words="english")
X_train = vectorizer.fit_transform(fit_texts)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, fit_labels)

train_time = time.time() - start_time


# ---------------------------------------------------------------------------
# Evaluate
# ---------------------------------------------------------------------------
X_test = vectorizer.transform(test_texts)
test_preds = clf.predict(X_test).tolist()
test_acc = accuracy_score(test_labels, test_preds)

print(f"Vocabulary size (TF-IDF features): {len(vectorizer.vocabulary_):,}")
print(f"Logistic regression coefficients: {clf.coef_.size:,}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Training time: {train_time:.2f}s")


# ---------------------------------------------------------------------------
# Save checkpoint + metrics
# ---------------------------------------------------------------------------
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_path = os.path.join(CHECKPOINT_DIR, "tfidf_spam.joblib")
joblib.dump({"vectorizer": vectorizer, "classifier": clf}, checkpoint_path)
checkpoint_size_mb = os.path.getsize(checkpoint_path) / (1024 ** 2)
print(f"Saved TF-IDF + LogisticRegression checkpoint to {checkpoint_path} "
      f"({checkpoint_size_mb:.4f} MB)")

os.makedirs(RESULTS_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, "tfidf_metrics.json"), "w") as f:
    json.dump({
        "strategy": "tfidf",
        "task": "spam",
        "test_accuracy": test_acc,
        "trainable_params": int(clf.coef_.size + clf.intercept_.size),
        "vocab_size": len(vectorizer.vocabulary_),
        "train_time_seconds": train_time,
        "checkpoint_path": checkpoint_path,
        "checkpoint_size_mb": checkpoint_size_mb,
        "test_predictions": test_preds,
        "test_labels": test_labels,
    }, f, indent=2)

print("Saved metrics to", os.path.join(RESULTS_DIR, "tfidf_metrics.json"))

Vocabulary size (TF-IDF features): 3,775
Logistic regression coefficients: 3,775
Test accuracy: 0.9467
Training time: 0.25s
Saved TF-IDF + LogisticRegression checkpoint to /content/drive/MyDrive/llm_finetune_project/checkpoints/spam/tfidf_spam.joblib (0.1600 MB)
Saved metrics to /content/drive/MyDrive/llm_finetune_project/results/spam/tfidf_metrics.json


## 4. Confirm results saved to Drive

In [4]:
!ls /content/drive/MyDrive/llm_finetune_project/results/spam/
!ls /content/drive/MyDrive/llm_finetune_project/checkpoints/spam/

frozen_metrics.json	    lora_metrics.json
full_finetune_metrics.json  tfidf_metrics.json
frozen_spam.pt	full_finetune_spam.pt  lora_spam.pt  tfidf_spam.joblib
